**Aim:** Differential Expression (DE) analysis comparing cells from different conditions (i.e. MDMA, Meth, Saline).
**Tools**
- `limma` – gene level 
- `iPAGE` – pathway level

# Differential analysis 

> Many methods have been used to determine differential gene expression from single-cell RNA (scRNA)-seq data. 

> Prefiltering of lowly expressed genes has important effects, particularly for some of the methods developed for bulk RNA-seq data analysis. However, we found that bulk RNA-seq analysis methods do not generally perform worse than those developed specifically for scRNA-seq. 

Source: [Soneson, C., Robinson, M. Bias, **robustness and scalability in single-cell differential expression analysis.** Nat Methods 15, 255–261 (2018).](https://doi.org/10.1038/nmeth.4612)

Also see 
[Wang, T., Li, B., Nelson, C.E. et al. **Comparative analysis of differential gene expression analysis tools for single-cell RNA sequencing data.** BMC Bioinformatics 20, 40 (2019).](https://doi.org/10.1186/s12859-019-2599-6)


[course chapter 12: Differential Expression (DE) analysis](https://biocellgen-public.svi.edu.au/mig_2019_scrnaseq-workshop/public/dechapter.html)


I'm also discussing this topic here https://github.com/abearab/sc-haircut-net/issues/3#issuecomment-832263091

<!-- > ### Estimate dispersion and DE using `DESeq2`
> Van den Berge and Perraudeau and others have shown the LRT may perform better for null hypothesis testing, so we use the LRT. In order to use the Wald test, it is recommended to set `useT=TRUE`.

> ### Plot dispersion estimates
> It is recommended to plot the dispersion estimates for DESeq2 on single-cell data. As discussed in the DESeq2 paper, it becomes difficult to accurately estimate the dispersion when the counts are very small, because the Poisson component of the variance is dominant. Therefore we see some very low dispersion estimates here, although the trend is still accurately capturing the upper proportion. So here everything looks good.


> If the parametric trend fails to fit (there would be a warning in this case), one should check the dispersion plot as above. If it looks like the dispersion fit is being thrown off by the low count genes with low dispersion estimates at the bottom of the plot, there is a relatively easy solution: one can filter out more of the low count genes only for the dispersion estimation step, so that the trend still captures the upper portion. This is pretty easy to do in DESeq2, to filter genes solely for the dispersion trend estimation, but to use a larger set for the rest of the analysis. An example of how this can be done:
 -->

___
Austen:
> Since mice were sacrificed 30 min after MDMA, Ting wondered what we should expect in terms of seeing differences in **immediate early gene expression (IEG)** versus other genes. After searching in the literature, IEG mRNA expression seems to peak after ~30 minutes, whereas other non-IEGs might take 1-3 hours to peak, depending on their length. Given this, we should probably focus on IEGs and shorter genes for the acute MDMA vs saline groups. Fortunately, GPCRs (indirect targets of MDMA) drive IEG expression (e.g., Fos, FosB, Egr1, jun, junb, and homer1 to name a few). That said, I think it would still be worth using SCENIC or KEGG to look at GPCR signaling pathways are modulated by MDMA.


In [1]:
import pandas as pd 
import scanpy as sc

In [2]:
def prep_diffexp_input(adata,name):
    counts = pd.DataFrame(
        adata.layers["counts"].T,
        columns = adata.obs.index.tolist(),
        index = adata.var.index.tolist()
    )

    meta = adata.obs[['batch','group']].copy()

    meta['treat'] = meta.batch.str.contains('acute').replace({True:'Acute',False:'notAcute'})
    meta['rep'] = meta.batch.str[-1]

    counts.to_csv(f'diffexp/{name}_counts.txt',  sep='\t')
    meta.to_csv(f'diffexp/{name}_meta.txt',sep='\t')

In [3]:
ls celltyping/*h5ad.gz

celltyping/adata_neuron-clustered.h5ad.gz
celltyping/adata_non_neuron-clustered-labeled.h5ad.gz


In [4]:
results_file = 'celltyping/adata_neuron-clustered.h5ad.gz'

adata_neuron = sc.read_h5ad(results_file)

In [5]:
results_file = 'celltyping/adata_non_neuron-clustered-labeled.h5ad.gz'

adata_non_neuron = sc.read_h5ad(results_file)

# Prep inputs

## neurons

In [8]:
set(adata_neuron.obs.leiden_neuron)

{'0', '1', '2', '3', '4', '5', '6', '7', '8'}

___
Austen:
> Please send me DEG lists (gene names, logfold(2) change, p values across treatments) for neuronal clusters 0, 3 , 4, and 5 so ic an explore volcano plots for these clusters that have all treatment groups present

### clust 0_3_4_5

In [9]:
adata_neuron_0_3_4_5 = adata_neuron[adata_neuron.obs.leiden_neuron.isin(['0','3','4','5'])].copy()

In [18]:
adata_neuron_0_3_4_5.obs.groupby(['group','leiden_neuron']).size()

group   leiden_neuron
MDMA    0                2734
        3                 564
        4                 363
        5                 339
Meth    0                 562
        3                 223
        4                  72
        5                  39
Saline  0                 621
        3                 190
        4                 204
        5                 104
dtype: int64

In [19]:
prep_diffexp_input(adata_neuron_0_3_4_5,'neuron_0_3_4_5')

### clust 0

In [21]:
adata_neuron_0 = adata_neuron[adata_neuron.obs.leiden_neuron.eq('0')].copy()

In [22]:
prep_diffexp_input(adata_neuron_0,'neuron_0')

### clust 3

In [23]:
adata_neuron_3 = adata_neuron[adata_neuron.obs.leiden_neuron.eq('3')].copy()

In [24]:
prep_diffexp_input(adata_neuron_3,'neuron_3')

### clust 4

In [26]:
adata_neuron_4 = adata_neuron[adata_neuron.obs.leiden_neuron.eq('4')].copy()

In [27]:
prep_diffexp_input(adata_neuron_4,'neuron_4')

### clust 5

In [28]:
adata_neuron_5 = adata_neuron[adata_neuron.obs.leiden_neuron.eq('5')].copy()

In [29]:
prep_diffexp_input(adata_neuron_5,'neuron_5')

## non-neuron

In [6]:
set(adata_non_neuron.obs.label)

{'Astrocytes', 'Endothelial', 'Microglia', 'Oligodendrocytes'}

### Astrocytes

In [16]:
Astrocytes = adata_non_neuron[adata_non_neuron.obs.label == 'Astrocytes'].copy()

In [17]:
prep_diffexp_input(Astrocytes,'Astrocytes')

### Microglia

In [18]:
Microglia = adata_non_neuron[adata_non_neuron.obs.label == 'Microglia'].copy()

In [19]:
prep_diffexp_input(Microglia,'Microglia')

### Endothelial


In [20]:
Endothelial = adata_non_neuron[adata_non_neuron.obs.label == 'Endothelial'].copy()

In [21]:
prep_diffexp_input(Endothelial,'Endothelial')

### Oligodendrocytes

In [22]:
Oligodendrocytes = adata_non_neuron[adata_non_neuron.obs.label == 'Oligodendrocytes'].copy()

In [23]:
prep_diffexp_input(Oligodendrocytes,'Oligodendrocytes')

## Run DE

___

`deseq2` env

## neurons

In [17]:
!Rscript diffexp.R all_neurons

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


### clust 0_3_4_5

In [1]:
!Rscript diffexp.R neuron_0_3_4_5

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


### clust 0

In [2]:
!Rscript diffexp.R neuron_0

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


### clust 3

In [3]:
!Rscript diffexp.R neuron_3

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


### clust 4

In [4]:
!Rscript diffexp.R neuron_4

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


### clust 5

In [5]:
!Rscript diffexp.R neuron_5

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


## non-neuron

### Astrocytes

In [18]:
!Rscript diffexp.R Astrocytes

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


### Microglia

In [19]:
!Rscript diffexp.R Microglia

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


### Endothelial


In [20]:
!Rscript diffexp.R Endothelial

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


### Oligodendrocytes

In [21]:
!Rscript diffexp.R Oligodendrocytes

---	load libraries	---
---	load counts	---
---	prep meta	---
---	run DE analysis	---


## Explore DE results

___

`deseq` R env

In [ ]:
source('plots.R')

draw_volcano <- function(name,xlim=10,lfc.cutoff = 1, pval.cutoff = 0.05){
    read.table(paste0('diffexp/',name,'/delta_exp_MDMAvsSaline.txt')) %>% 
        plot_Volcano (x_min=-xlim,x_max=xlim,lfc.cutoff = lfc.cutoff, pval.cutoff = pval.cutoff ) +
        ggtitle(paste0(name,'\nMDMAvsSaline'))  -> p1

    read.table(paste0('diffexp/',name,'/delta_exp_MDMAvsMeth.txt')) %>% 
        plot_Volcano (x_min=-xlim,x_max=xlim,lfc.cutoff = lfc.cutoff, pval.cutoff = pval.cutoff ) +
        ggtitle('MDMAvsMeth')  -> p2

    read.table(paste0('diffexp/',name,'/delta_exp_MethvsSaline.txt')) %>% 
        plot_Volcano (x_min=-xlim,x_max=xlim,lfc.cutoff = lfc.cutoff, pval.cutoff = pval.cutoff ) +
        ggtitle('MethvsSaline')  -> p3

    return(p1 / p2/ p3)
}

In [ ]:
draw_volcano('all_neurons',xlim=5,lfc.cutoff=0.5)

### Astrocytes

In [ ]:
draw_volcano('Astrocytes',5)

### Microglia

In [ ]:
draw_volcano('Microglia',5)

### Endothelial


In [ ]:
draw_volcano('Endothelial',5)

### Oligodendrocytes

In [ ]:
draw_volcano('Oligodendrocytes',xlim=5,lfc.cutoff=0.8)

## Pathway analysis

___

`down` R env

In [1]:
import pandas as pd 
# from utils import *

from matplotlib import pyplot as plt
from venn import venn 

In [2]:
import blitzgsea as blitz

In [3]:
def read_diffexp_table(name):
    d = {
        'Meth_vs_Saline':pd.read_csv(
            f'diffexp/{name}/delta_exp_MethvsSaline.txt',  sep='\t'),
        'MDMA_vs_Saline':pd.read_csv(
            f'diffexp/{name}/delta_exp_MDMAvsSaline.txt',sep='\t'),
        'MDMA_vs_Meth':pd.read_csv(
            f'diffexp/{name}/delta_exp_MDMAvsMeth.txt',  sep='\t'),
        'MDMA_Acute_vs_notAcute':pd.read_csv(
            f'diffexp/{name}/delta_exp_MDMA_Acute_vs_notAcute.txt',  sep='\t')
    }

    out = pd.concat(d.values(), axis=1, keys=d.keys())
    out.index = out.index.str.upper()
    return out

### Load gene sets

In [8]:
m5_gmt = blitz.enrichr.read_gmt(
    # '/data_gilbert/home/aarab/tools/pager/annotations/mouse/m5.all.v2022.1.Mm.symbols.gmt'
    '/data_gilbert/home/aarab/tools/pager/annotations/mouse/m5.go.v2023.1.Mm.symbols.gmt'
)

In [10]:
syngo = pd.read_csv('../../synGOpy/syngo_ontologies_clean.txt',sep='\t').set_index('name').rename(columns={'hgnc_symbol':'genes'})

syngo_gmt = syngo.genes.str.split(', ').to_dict()

___

### Run GSEA

In [13]:
GSEA_res = {}

for cond in ['all_neurons',
             'neuron_0_3_4_5','neuron_0','neuron_3','neuron_4','neuron_5',
             'Astrocytes', 'Endothelial', 'Microglia', 'Oligodendrocytes']:
    GSEA_res[cond] = {}
    for comp in ['MDMA_vs_Meth','MDMA_vs_Saline','Meth_vs_Saline','MDMA_Acute_vs_notAcute']:
        signature = read_diffexp_table(cond)[comp]['logFC'].reset_index()
        result1 = blitz.gsea(signature,m5_gmt,min_size=15,max_size=150,verbose=True)
        result2 = blitz.gsea(signature,syngo_gmt,verbose=True)
        
        GSEA_res[cond][comp] = {
            'signature':signature,
            'result_GO':result1,
            'result_synGO':result2,
        }

Enrichment : 100%|██████████| 10673/10673 [00:04<00:00, 2366.07it/s]
/data_gilbert/home/aarab/anaconda3/envs/down/lib/python3.10/site-packages/statsmodels/stats/multitest.py:155: RuntimeWarning: divide by zero encountered in log1p
  pvals_corrected = -np.expm1(ntests * np.log1p(-pvals))
Enrichment : 100%|██████████| 293/293 [00:00<00:00, 1127.65it/s]
/data_gilbert/home/aarab/anaconda3/envs/down/lib/python3.10/site-packages/statsmodels/stats/multitest.py:155: RuntimeWarning: divide by zero encountered in log1p
  pvals_corrected = -np.expm1(ntests * np.log1p(-pvals))
Enrichment : 100%|██████████| 10673/10673 [00:04<00:00, 2330.99it/s]
/data_gilbert/home/aarab/anaconda3/envs/down/lib/python3.10/site-packages/statsmodels/stats/multitest.py:155: RuntimeWarning: divide by zero encountered in log1p
  pvals_corrected = -np.expm1(ntests * np.log1p(-pvals))
Enrichment : 100%|██████████| 293/293 [00:00<00:00, 1151.19it/s]
/data_gilbert/home/aarab/anaconda3/envs/down/lib/python3.10/site-packages/s

___
Save DE genes and gene ontologies into excel files:

In [14]:
date = '2023_03_27'

for cond in ['all_neurons', 
             'neuron_0_3_4_5','neuron_0','neuron_3','neuron_4','neuron_5',
             'Astrocytes', 'Endothelial', 'Microglia', 'Oligodendrocytes']:
    
    writer = pd.ExcelWriter(f'diffexp/{date}_DEG_{cond}.xlsx', engine='xlsxwriter')

    pd.concat(dict([
        (comp,read_diffexp_table(cond)[comp][['logFC','P.Value','adj.P.Val']])
        for comp in ['MDMA_vs_Meth','MDMA_vs_Saline','Meth_vs_Saline','MDMA_Acute_vs_notAcute']
    ]),axis=1).to_excel(writer, sheet_name='DE_genes')
    
    for comp in ['MDMA_vs_Meth','MDMA_vs_Saline','Meth_vs_Saline','MDMA_Acute_vs_notAcute']:
        GSEA_res[cond][comp]['result_GO'].to_excel(
            writer, sheet_name=f'{comp}_GO'
        )
        
        GSEA_res[cond][comp]['result_synGO'].to_excel(
            writer, sheet_name=f'{comp}_synGO'
        )
    writer.save()

/tmp/ipykernel_86904/641997964.py:22: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer.save()
/tmp/ipykernel_86904/641997964.py:22: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer.save()
/tmp/ipykernel_86904/641997964.py:22: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer.save()
/tmp/ipykernel_86904/641997964.py:22: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer.save()
/tmp/ipykernel_86904/641997964.py:22: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer.save()
/tmp/ipykernel_86904/641997964.py:22: FutureWarning: save is not part of the public API, usage can give u

___

In [11]:
# signature = GSEA_res['Oligodendrocytes']['MDMAvsMeth']['signature'].copy()
# gsea_result = GSEA_res['Oligodendrocytes']['MDMAvsMeth']['result_GO'].copy()

# fig_table = blitz.plot.top_table(
#     signature,m5_gmt,
#     gsea_result[
#         ((gsea_result.index.str.contains('NEURO')) | 
#         (gsea_result.index.str.contains('BRAIN'))) & 
#         (gsea_result.geneset_size < 50)
#          ]
# )
# # fig_table.savefig("gsea_top_nes_pos_table.pdf", bbox_inches='tight')
# fig_table

### 

In [12]:
# %%bash
# zip -r diffexp/Astrocytes_DE.zip diffexp/Astrocytes_delta_exp_* 

# zip -r diffexp/Microglia_DE.zip diffexp/Microglia_delta_exp_* 

#

In [13]:
!date

Wed Dec  7 00:41:07 PST 2022
